# Notebook 20 — Phase-Lock Order Parameter

This notebook replaces threshold-based survival with a single smooth diagnostic:

**phase-lock order parameter**
\[
S = F_{CZ} \cdot C \cdot (1 - L)
\]

where:
- \(F_{CZ}\) is the compensated CZ fidelity,
- \(C\) is a coherence proxy,
- \(L\) is a leakage proxy.

This gives a continuous phase diagram for where the phase-locked CZ structure remains usable under noise.

It is the natural follow-up to Notebook 19:
- Notebook 19 used thresholds to define a binary survival region.
- Notebook 20 builds a **continuous order parameter** instead.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Phase-locked protocol from Notebook 17d / 18b / 19

In [ ]:
opt = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

for k, v in opt.items():
    print(f"{k}: {v:.6f}")


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, T, omega_max, alpha, V, delta0, p, q,
                        gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    H = build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q)
    times = np.linspace(0.0, T, n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(T, omega_max, alpha, V, delta0, p, q,
                              gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, T, omega_max, alpha, V, delta0, p, q,
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## 2D noise scan

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 25)
gamma_phi_vals = np.linspace(0.0, 0.12, 25)

cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        U_eff, finals = build_noisy_effective_map(
            opt['T'], opt['omega_max'], opt['alpha'], V, opt['delta0'], opt['p'], opt['q'],
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=200
        )
        cz_map[i, j] = compensated_cz_fidelity(U_eff)
        coh_map[i, j] = coherence_proxy(finals)
        leak_map[i, j] = leakage_from_finals(finals)


## Phase-lock order parameter

In [ ]:
phase_lock_order = cz_map * coh_map * (1.0 - leak_map)

print("Order parameter summary:")
print(f"max S = {phase_lock_order.max():.6f}")
print(f"min S = {phase_lock_order.min():.6f}")


## Plot continuous phase-lock order parameter

In [ ]:
plt.figure(figsize=(7.2, 5.4))
im = plt.imshow(
    phase_lock_order,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
)
levels = [0.002, 0.005, 0.01, 0.02]
plt.contour(gamma_decay_vals, gamma_phi_vals, phase_lock_order, levels=levels, colors='white', linewidths=1.0)
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Pure dephasing gamma_phi')
plt.title('Phase-lock order parameter')
plt.colorbar(im, label='S')
plt.tight_layout()
plt.show()


## Soft boundary from order parameter

In [ ]:
boundary_level = 0.005

plt.figure(figsize=(7.2, 5.4))
im = plt.imshow(
    phase_lock_order,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
)
plt.contour(gamma_decay_vals, gamma_phi_vals, phase_lock_order, levels=[boundary_level], colors='white', linewidths=1.5)
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Pure dephasing gamma_phi')
plt.title(f'Phase-lock region (S > {boundary_level})')
plt.colorbar(im, label='S')
plt.tight_layout()
plt.show()


## Compare order parameter against core ingredients

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))

im0 = axes[0].imshow(
    phase_lock_order, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[0].contour(gamma_decay_vals, gamma_phi_vals, phase_lock_order, levels=[boundary_level], colors='white', linewidths=1.0)
axes[0].set_title('Order parameter S')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('gamma_phi')

im1 = axes[1].imshow(
    cz_map, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[1].contour(gamma_decay_vals, gamma_phi_vals, phase_lock_order, levels=[boundary_level], colors='white', linewidths=1.0)
axes[1].set_title('Compensated CZ fidelity')
axes[1].set_xlabel('gamma')
axes[1].set_ylabel('gamma_phi')

im2 = axes[2].imshow(
    coh_map, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[2].contour(gamma_decay_vals, gamma_phi_vals, phase_lock_order, levels=[boundary_level], colors='white', linewidths=1.0)
axes[2].set_title('Coherence proxy')
axes[2].set_xlabel('gamma')
axes[2].set_ylabel('gamma_phi')

im3 = axes[3].imshow(
    leak_map, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[3].contour(gamma_decay_vals, gamma_phi_vals, phase_lock_order, levels=[boundary_level], colors='white', linewidths=1.0)
axes[3].set_title('Leakage proxy')
axes[3].set_xlabel('gamma')
axes[3].set_ylabel('gamma_phi')

fig.colorbar(im0, ax=axes[0], label='S')
fig.colorbar(im1, ax=axes[1], label='F_CZ')
fig.colorbar(im2, ax=axes[2], label='C')
fig.colorbar(im3, ax=axes[3], label='L')
plt.tight_layout()
plt.show()


## 1D slices of the order parameter

In [ ]:
S_gamma = phase_lock_order[0, :]   # gamma_phi = 0
S_phi = phase_lock_order[:, 0]       # gamma = 0


In [ ]:
plt.figure(figsize=(7, 4.4))
plt.plot(gamma_decay_vals, S_gamma)
plt.axhline(boundary_level, linestyle='--', label='soft boundary')
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('S')
plt.title('Phase-lock score at gamma_phi = 0')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4.4))
plt.plot(gamma_phi_vals, S_phi)
plt.axhline(boundary_level, linestyle='--', label='soft boundary')
plt.xlabel('Pure dephasing gamma_phi')
plt.ylabel('S')
plt.title('Phase-lock score at gamma = 0')
plt.legend()
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
gamma_axis_survive = gamma_decay_vals[S_gamma > boundary_level]
gamma_phi_axis_survive = gamma_phi_vals[S_phi > boundary_level]

gamma_max_survive = float(gamma_axis_survive.max()) if gamma_axis_survive.size else 0.0
gamma_phi_max_survive = float(gamma_phi_axis_survive.max()) if gamma_phi_axis_survive.size else 0.0

summary_text = f'''
Phase-lock order-parameter summary

Protocol:
T      = {opt["T"]:.4f}
alpha  = {opt["alpha"]:.4f}
Omega  = {opt["omega_max"]:.4f}
Delta0 = {opt["delta0"]:.4f}
p      = {opt["p"]:.4f}
q      = {opt["q"]:.4f}

Order parameter:
S = F_CZ * C * (1 - L)

Global summary:
max S = {phase_lock_order.max():.6f}
min S = {phase_lock_order.min():.6f}
soft boundary level = {boundary_level:.6f}

Axis survival estimates:
max gamma surviving at gamma_phi = 0  -> {gamma_max_survive:.4f}
max gamma_phi surviving at gamma = 0  -> {gamma_phi_max_survive:.4f}

Interpretation:
- high S means simultaneously high compensated CZ fidelity, strong coherence, and low leakage,
- S decays quickly away from the low-noise corner,
- the surviving phase-locked CZ region is bounded and small in this simple Lindblad model.
'''
print(summary_text)


## Final conclusion

This notebook replaces threshold logic with a single continuous phase-lock order parameter.

That gives the cleanest final phase diagram in the repo:

- a smooth map of where the phase-locked CZ gate exists,
- a soft survival boundary,
- and a compact summary of how far that region extends along the main noise axes.

This supports the final statement:

**the phase-locked shaped CZ solution is a sharply bounded low-noise phenomenon whose survival is controlled jointly by gate fidelity, coherence retention, and leakage suppression.**
